In [ ]:
import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [ ]:
data = pd.read_excel("sample data.xlsx")

In [ ]:
# combine all text from interview questions into one col
text_cols = [f"Q{i}_CH" for i in range(1, 34) if i != 22]
data["combined_text"] = (data[text_cols].fillna("").astype(str).agg(" ".join, axis=1).str.strip())

# for this demo, just select 1 trait
targetvariable = "Extroversion_CH"
data[targetvariable] = pd.to_numeric(data[targetvariable], errors="coerce")

data[targetvariable] = pd.to_numeric(data[targetvariable], errors="coerce")
data_1 = data.loc[data[targetvariable].notna() & data["combined_text"].ne("")].copy()
data_1.head()

In [ ]:
# Test/Train split (50% training/20% test) 
train, test = train_test_split(data_1, test_size=0.20, random_state=806)

vectorizer = TfidfVectorizer(min_df=5,max_features=2000,stop_words="english")

X_train = vectorizer.fit_transform(train["combined_text"]).toarray()
X_test = vectorizer.transform(test["combined_text"]).toarray()

# Select the personality scores to predict
y_train = train[targetvariable]
y_test = test[targetvariable]

feature_names = vectorizer.get_feature_names_out()

print(f"Training participants: {len(train)}; test participants: {len(test)}")
print(f"Word features: {len(feature_names)}")

In [ ]:
# Train the model
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

# Evaluate performance
print(f"Test R²: {r2_score(y_test, predictions):.3f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.3f}")

# Compare with simply predicting the training-set mean
baseline = np.full(len(y_test), y_train.mean())
print(f"Baseline RMSE: {np.sqrt(mean_squared_error(y_test, baseline)):.3f}")

In [ ]:
masker = shap.maskers.Independent(X_train, max_samples=len(X_train))
explainer = shap.LinearExplainer(model, masker, feature_names=feature_names)
shap_values = explainer(X_test)
shap.plots.bar(shap_values, max_display=20)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=20)